# Chapter 14 &mdash; Why Study Impossibility Results?

**Concept 1 of the Chapter 14 decomposition:** *Why Study Impossibility Results?*

Knowing what a machine <i>cannot</i> do is as informative as knowing what it can.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Why-Impossibility-Results/Concept-Why-Impossibility-Results.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Impossibility results are not pessimism; they are **engineering information**.

* They tell you **when to stop looking.** No amount of cleverness yields a perfect
  static analyser for "does this program crash", so tool builders aim for *sound but
  incomplete* or *complete but unsound* instead.
* They **redirect effort.** Type systems, model checkers, and linters are all answers
  to "the general question is undecidable, so what restricted question is not?"
* They are **robust.** Unlike performance claims, they do not expire when hardware
  improves.

The pattern to internalise: *the general problem is undecidable; a decidable
approximation is what ships*.

## 2. Definitions

### A decidable question and its undecidable big brother

In [ ]:
def re_dfa_emptiness(D):
    # DECIDABLE: is L(D) empty?  Reachability from q0 to any final state.
    seen, frontier = {D["q0"]}, {D["q0"]}
    while frontier:
        nxt = {step_dfa(D, q, a) for q in frontier for a in D["Sigma"]} - seen
        seen |= nxt; frontier = nxt
    return not (seen & D["F"])

### The approximation pattern, spelled out

In [ ]:
STANCES = [("sound, incomplete", "never wrong when it says 'safe'; may refuse to decide",
            "type systems, most static analysers"),
           ("complete, unsound", "finds every bug; also reports non-bugs",
            "aggressive linters, some fuzz triage"),
           ("bounded",           "decides correctly up to a size or depth limit",
            "bounded model checking, SAT-based tools")]

## 3. Tests

DFA emptiness **is** decidable, and the algorithm always terminates.

In [ ]:
Empty = md2mc('''DFA
I : 0 | 1 -> I
''')
NotEmpty = md2mc('''DFA
IF : 0 | 1 -> IF
''')
print("L(Empty)    is empty?", re_dfa_emptiness(Empty))
print("L(NotEmpty) is empty?", re_dfa_emptiness(NotEmpty))
assert re_dfa_emptiness(Empty) and not re_dfa_emptiness(NotEmpty)

It terminates on **every** DFA, which is what 'decidable' means.

In [ ]:
import random
for trial in range(20):
    n = random.randint(1, 6)
    names = ['I'] + ['S%d' % i for i in range(1, n)]
    fin = random.sample(names, random.randint(0, n))
    lines = ['DFA']
    for q in names:
        for a in '01':
            t = random.choice(names)
            lines.append('%s : %s -> %s' % (('F' + q if q in fin and q != 'I'
                                             else ('IF' if q == 'I' and q in fin else q)),
                                            a, ('F' + t if t in fin and t != 'I'
                                                else ('IF' if t == 'I' and t in fin else t))))
    D = md2mc('\n'.join(lines))
    re_dfa_emptiness(D)      # must simply return, every time
print("emptiness decided for 20 random DFA, no timeouts, no fuel")

The same question about **programs** is not decidable &mdash; so tools approximate.

In [ ]:
print("%-20s %-52s %s" % ("stance", "guarantee", "examples"))
for a, b, d in STANCES:
    print("%-20s %-52s %s" % (a, b, d))

What impossibility buys you, concretely.

In [ ]:
USES = ["stop searching for the perfect tool -- it does not exist",
        "choose your unsoundness deliberately rather than by accident",
        "explain to a manager why 'just detect all bugs' is not a backlog item",
        "recognise a reduction when a new problem is the old one in disguise"]
for u in USES: print("  *", u)

## 4. Exercises


1. Name a tool you use daily that is deliberately incomplete. What does it give up?
2. Is "does this regular expression match anything?" decidable? Why?
3. Which of the three stances would you pick for a compiler warning? For a verifier?

In [ ]:
# Your work for the exercises above.